# 07 · Ingestão Gold — Squad 3 | Henrique Ficht
Lê as tabelas Delta da camada **silver**, aplica regras de negócio,
sumariza/agrega e salva como **Delta** na camada **gold** e também no
**SQL Server** (schema `squad3`) para consumo via Looker.

- **Modo de escrita Delta:** `merge` (upsert por PK) para tabelas acumulativas;
  `overwrite` para tabelas de alerta (L6, L10)
- **Modo de escrita SQL Server:** staging + `MERGE` T-SQL (upsert);
  `TRUNCATE + INSERT` para tabelas de alerta
- **Dependências de Silver:** fallback para Raw aceito temporariamente para
  `physical_produtos_pereciveis` e `food_estoque_lojas` até Silvers serem publicadas
- **Frequência:** semanal (após 06)

### Tabelas Gold produzidas (10)

**Fornecedores (4)**
- `gold_fornecedor_lead_time_trimestre` (F6) — MERGE
- `gold_fornecedor_ativos_categoria` (F7+F8) — MERGE
- `gold_fornecedor_distribuicao_uf` (F9) — MERGE
- `gold_fornecedor_tempo_ativacao` (F10) — MERGE

**Lotes (5)**
- `gold_lote_alerta_ativo_vencido` (L6) — OVERWRITE
- `gold_lote_volume_categoria_mes` (L7) — MERGE
- `gold_lote_recall_fornecedor_trimestre` (L8) — MERGE
- `gold_lote_shelf_life_categoria` (L9) — MERGE
- `gold_lote_sem_estoque` (L10) — OVERWRITE

**Dimensão auxiliar (1)**
- `gold_fornecedor_canal_distribuicao` (L10 — perfil) — MERGE

### Sprint atual
**Sprint 1** — Setup + Leitura + Enriquecimento base + Funções de gravação (sem construir tabelas Gold ainda).

## BLOCO 1 — Setup

In [0]:
%pip install python-dotenv --quiet
dbutils.library.restartPython()

In [0]:
import os
from dotenv import load_dotenv

ENV_PATH = "/Workspace/Users/ik.kukoo@gmail.com/.env"
load_dotenv(ENV_PATH, override=True)

def _require(key):
    value = os.getenv(key)
    if not value:
        raise EnvironmentError(f"Variável '{key}' não encontrada em {ENV_PATH}")
    return value

STORAGE_ACCOUNT = "internshipdatalake"

# ── ADLS Gen2 (OAuth / Service Principal) ───────────────────────────────
adls_options = {
    f"fs.azure.account.auth.type.{STORAGE_ACCOUNT}.dfs.core.windows.net": "OAuth",
    f"fs.azure.account.oauth.provider.type.{STORAGE_ACCOUNT}.dfs.core.windows.net":
        "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider",
    f"fs.azure.account.oauth2.client.id.{STORAGE_ACCOUNT}.dfs.core.windows.net":
        _require("ADLS_CLIENT_ID"),
    f"fs.azure.account.oauth2.client.secret.{STORAGE_ACCOUNT}.dfs.core.windows.net":
        _require("ADLS_CLIENT_SECRET"),
    f"fs.azure.account.oauth2.client.endpoint.{STORAGE_ACCOUNT}.dfs.core.windows.net":
        f"https://login.microsoftonline.com/{_require('ADLS_TENANT_ID')}/oauth2/token",
}

# ── Paths ADLS ──────────────────────────────────────────────────────────
RAW_BASE      = f"abfss://raw@{STORAGE_ACCOUNT}.dfs.core.windows.net/batch-data"
SILVER_BASE   = f"abfss://squad3@{STORAGE_ACCOUNT}.dfs.core.windows.net/silver"
GOLD_BASE     = f"abfss://squad3@{STORAGE_ACCOUNT}.dfs.core.windows.net/gold"
SILVER_SQUAD1 = f"abfss://squad1@{STORAGE_ACCOUNT}.dfs.core.windows.net/silver"

# ── SQL Server (Azure) ──────────────────────────────────────────────────
SQL_HOST = _require("SQL_HOST")
SQL_DB   = _require("SQL_DATABASE")
SQL_USER = _require("SQL_USERNAME")
SQL_PASS = _require("SQL_PASSWORD")

# Opções do conector nativo "sqlserver" do Databricks Serverless
sqlserver_opts = {
    "host":                   SQL_HOST,
    "port":                   "1433",
    "database":               SQL_DB,
    "user":                   SQL_USER,
    "password":               SQL_PASS,
    "encrypt":                "true",
    "trustServerCertificate": "false",
}

print("✅ Config carregado.")
print(f"   SILVER → {SILVER_BASE}")
print(f"   GOLD   → {GOLD_BASE}")
print(f"   SQL    → {SQL_HOST} / {SQL_DB}")

In [0]:
# ── Widget: data de referência para L6 (alerta ativo vencido) ───────────
# Default vazio → usa current_date().
# Aceita formatos: YYYY-MM-DD (ISO) ou DD/MM/YYYY (BR).
# Permite backtest: passar '2025-06-30' para simular alerta naquela data.
from pyspark.sql.functions import current_date, lit, to_date, coalesce

dbutils.widgets.text(
    "data_referencia",
    "",
    "Data de referência (YYYY-MM-DD ou DD/MM/YYYY, vazio = hoje)"
)

_data_ref_raw = dbutils.widgets.get("data_referencia").strip()

if _data_ref_raw:
    # Tenta ISO primeiro, depois formato BR
    DATA_REFERENCIA = coalesce(
        to_date(lit(_data_ref_raw), "yyyy-MM-dd"),
        to_date(lit(_data_ref_raw), "dd/MM/yyyy"),
    )
    # Força avaliação para falhar aqui, não lá na frente, se o formato for inválido
    _teste = spark.range(1).select(DATA_REFERENCIA.alias("data_ref")).collect()[0]["data_ref"]
    if _teste is None:
        raise ValueError(
            f"Data '{_data_ref_raw}' não é válida. "
            f"Use YYYY-MM-DD (ex: 2026-10-02) ou DD/MM/YYYY (ex: 02/10/2026)."
        )
    print(f"✅ Data de referência (manual): {_data_ref_raw} → interpretado como {_teste}")
else:
    DATA_REFERENCIA = current_date()
    print(f"✅ Data de referência: current_date() (hoje)")

## BLOCO 2 — Leitura das Silvers

- Silvers principais da Squad 3 (`food_fornecedores`, `food_lotes_producao`): leitura rígida — se faltar, falha.
- Silvers de referência (`physical_produtos_pereciveis`, `ecommerce_categorias`, `ecommerce_produtos`, `food_estoque_lojas`): fallback Silver→Raw aceito temporariamente, com log explícito.

In [0]:
from pyspark.sql.functions import col, trim

df_fornecedores = (
    spark.read.format("delta").options(**adls_options)
    .load(f"{SILVER_BASE}/food_fornecedores")
)
df_lotes = (
    spark.read.format("delta").options(**adls_options)
    .load(f"{SILVER_BASE}/food_lotes_producao")
)

print(f"✅ food_fornecedores:   {df_fornecedores.count():>6} registros | {len(df_fornecedores.columns)} colunas")
print(f"✅ food_lotes_producao: {df_lotes.count():>6} registros | {len(df_lotes.columns)} colunas")

In [0]:
# ── Helper: tenta Silver, cai pro Raw se não existir ────────────────────
# IMPORTANTE: força .count() dentro do try para lazy evaluation acusar o erro aqui
origens = {}

def read_silver_or_raw(label, silver_path, raw_path):
    try:
        df = spark.read.format("delta").options(**adls_options).load(silver_path)
        df.count()  # força resolução do path
        origens[label] = "silver"
        print(f"✅ {label:<32s} SILVER → {df.count():>6} registros")
        return df
    except Exception:
        df = (
            spark.read.options(**adls_options)
            .option("header", "true").option("inferSchema", "true")
            .csv(raw_path)
        )
        origens[label] = "raw"
        print(f"⚠️  {label:<32s} RAW    → {df.count():>6} registros (sem flags)")
        return df

# ── Catálogo de SKUs perecíveis + categoria_pai (para L7, L9, L10) ──────
df_pereciveis = read_silver_or_raw(
    "physical_produtos_pereciveis",
    f"{SILVER_BASE}/physical_produtos_pereciveis",
    f"{RAW_BASE}/physical_produtos_pereciveis.csv",
)

# ── Catálogo de categorias e-commerce (para F7+F8) ──────────────────────
df_ecom_categ = read_silver_or_raw(
    "ecommerce_categorias",
    f"{SILVER_SQUAD1}/ecommerce_categorias",
    f"{RAW_BASE}/ecommerce_categorias.csv",
)

# ── Catálogo de SKUs e-commerce (para L7) ───────────────────────────────
df_ecom_produtos = read_silver_or_raw(
    "ecommerce_produtos",
    f"{SILVER_SQUAD1}/ecommerce_produtos",
    f"{RAW_BASE}/ecommerce_produtos.csv",
)

# ── Estoque das lojas (para L10) ────────────────────────────────────────
df_estoque = read_silver_or_raw(
    "food_estoque_lojas",
    f"{SILVER_BASE}/food_estoque_lojas",
    f"{RAW_BASE}/food_estoque_lojas.csv",
)

print()
print("── Origens das Silvers de referência ──")
for k, v in origens.items():
    icone = "✅" if v == "silver" else "⚠️"
    print(f"   {icone} {k:<32s} ← {v}")

silvers_pendentes = [k for k, v in origens.items() if v == "raw"]
if silvers_pendentes:
    print(f"\n⚠️  Débito técnico: {len(silvers_pendentes)} Silver(s) ainda pendente(s) de publicação")
    print(f"   Quando publicadas, esta Gold passará a lê-las direto da Silver.")

## BLOCO 3 — Enriquecimento base

DataFrames reutilizados por várias tabelas Gold.

In [0]:
# ── Catálogo auxiliar: SKU → categoria_pai (perecíveis) ─────────────────
pereciveis_cat = (
    df_pereciveis
    .select(trim(col("sku")).alias("sku"),
            col("categoria_pai"))
    .distinct()
)

# ── Catálogo auxiliar: id_fornecedor → razao_social, tipo_fornecedor ────
fornecedores_dim = (
    df_fornecedores
    .select("id_fornecedor", "razao_social", "tipo_fornecedor", "is_ativo")
)

# ── df_lotes_enriq: lotes + categoria_pai (left join) + razao_social ────
# Left join com perecíveis: lotes de e-commerce ficam com categoria_pai NULL
# (serão agrupados como 'E-commerce (não perecível)' nas Golds de L7)
# torna recomputação barata; se virar gargalo nos próximos sprints,
# considerar .localCheckpoint() ou materializar em tabela temporária.
df_lotes_enriq = (
    df_lotes
    .withColumn("sku", trim(col("sku")))
    .join(pereciveis_cat, "sku", "left")
    .join(fornecedores_dim, "id_fornecedor", "left")
)

# ── df_fornecedores_limpos: sem flags de tipo/lead_time inválidos ───────
# Usado nas tabelas que agregam por tipo_fornecedor ou lead_time (F6)
df_fornecedores_limpos = (
    df_fornecedores
    .filter(~col("_flag_tipo_invalido") & ~col("_flag_lead_time_invalido"))
)

# ── Reportar métricas base ──────────────────────────────────────────────
qtd_enriq  = df_lotes_enriq.count()
qtd_limpos = df_fornecedores_limpos.count()
qtd_com_cat = df_lotes_enriq.filter(col("categoria_pai").isNotNull()).count()

print(f"✅ df_lotes_enriq:          {qtd_enriq} registros")
print(f"   com categoria_pai:       {qtd_com_cat} ({qtd_com_cat/qtd_enriq*100:.1f}%)")
print(f"   sem categoria_pai:       {qtd_enriq - qtd_com_cat} ({(qtd_enriq-qtd_com_cat)/qtd_enriq*100:.1f}% — e-commerce)")
print(f"✅ df_fornecedores_limpos:  {qtd_limpos} registros ({df_fornecedores.count() - qtd_limpos} excluídos por flags)")

## BLOCO 4 — Funções genéricas de gravação

Quatro funções, duas para Delta e duas para SQL Server:
- `save_gold_delta_merge` — PK composta, upsert por `gold_processed_at`
- `save_gold_delta_overwrite` — snapshot do estado atual (L6, L10)
- `save_gold_sqlserver_merge` — staging + MERGE T-SQL via JDBC
- `save_gold_sqlserver_overwrite` — TRUNCATE + INSERT via JDBC

**Convenção:** Delta primeiro, SQL depois. Se o SQL falhar, re-roda sem retrabalho.

In [0]:
from pyspark.sql.functions import row_number, current_timestamp
from pyspark.sql.window import Window

In [0]:
def save_gold_delta_merge(df_new, gold_path, pk_cols, table_name):
    """
    MERGE manual: lê Delta existente, une com df_new, deduplica por PK
    (mantém mais recente por gold_processed_at) e sobrescreve.

    Compatível com Databricks Serverless + Unity Catalog, que bloqueia
    DeltaTable.merge() por restrição de spark.conf.set() para Hadoop.

    pk_cols: str ou list[str] (suporta PK composta)
    """
    pk_list = [pk_cols] if isinstance(pk_cols, str) else pk_cols

    print(f"\n{'='*60}")
    print(f"💾 Delta MERGE: {table_name}")
    print(f"   Path: {gold_path}")
    print(f"   PK:   {pk_list}")
    print(f"{'='*60}")

    try:
        df_existing = (
            spark.read.format("delta")
            .options(**adls_options)
            .load(gold_path)
        )
        existing_count = df_existing.count()
        print(f"   📂 Delta existente: {existing_count} registros")
        df_merged = df_existing.unionByName(df_new, allowMissingColumns=True)
    except Exception:
        print(f"   📂 Delta não existe → escrita inicial")
        df_merged = df_new

    # Deduplicar por PK, mantendo o mais recente
    w = Window.partitionBy(*pk_list).orderBy(col("gold_processed_at").desc())
    df_final = (
        df_merged
        .withColumn("_rn", row_number().over(w))
        .filter(col("_rn") == 1)
        .drop("_rn")
    )

    writer = (
        df_final.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .options(**adls_options)
    )

    # Particionar por 'ano' se existir no schema
    if "ano" in df_final.columns:
        writer = writer.partitionBy("ano")

    writer.save(gold_path)

    final_count = df_final.count()
    print(f"   ✅ Gravação concluída: {final_count} registros")
    return df_final

print("✅ Função save_gold_delta_merge definida.")

In [0]:
def save_gold_delta_overwrite(df, gold_path, table_name):
    """
    Overwrite completo — snapshot do estado atual.
    Usado para tabelas de alerta (L6, L10) onde registros podem SAIR do
    universo entre execuções (ex: lote vencido que foi baixado do ERP).
    MERGE não serve aqui porque chaves antigas devem ser removidas.
    """
    print(f"\n{'='*60}")
    print(f"💾 Delta OVERWRITE: {table_name}")
    print(f"   Path: {gold_path}")
    print(f"{'='*60}")

    writer = (
        df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .options(**adls_options)
    )

    if "ano" in df.columns:
        writer = writer.partitionBy("ano")

    writer.save(gold_path)

    count = df.count()
    print(f"   ✅ Gravação concluída: {count} registros")
    return df

print("✅ Função save_gold_delta_overwrite definida.")

In [0]:
def _execute_sql(sql_statements):
    """
    Executa comandos SQL no SQL Server usando o Spark JDBC Reader via gRPC
    (compatível com Databricks Serverless / Spark Connect).
    
    Alternativa a spark._jvm.java.sql.DriverManager, que não funciona no
    Serverless porque não há Py4J bridge direto.
    
    Técnica: usa a opção `sessionInitStatement` do JDBC Reader para rodar
    o SQL antes da query de leitura. A query de leitura é um SELECT 1
    descartável, só para satisfazer o requisito de ResultSet do reader.
    
    sql_statements: str (um comando) ou list[str] (vários, concatenados)
    """
    if isinstance(sql_statements, str):
        sql_statements = [sql_statements]
    
    # Concatena os comandos em um único batch (separados por ;)
    # SQL Server aceita múltiplos comandos no sessionInitStatement
    batch_sql = "\n".join(
        s.strip().rstrip(";") + ";" for s in sql_statements if s.strip()
    )
    
    jdbc_url = (
        f"jdbc:sqlserver://{SQL_HOST}:1433;"
        f"database={SQL_DB};"
        "encrypt=true;trustServerCertificate=false;"
        "hostNameInCertificate=*.database.windows.net;loginTimeout=30;"
    )
    
    # Força execução do sessionInitStatement + query descartável
    _ = (
        spark.read.format("jdbc")
        .option("url", jdbc_url)
        .option("user", SQL_USER)
        .option("password", SQL_PASS)
        .option("driver", "com.microsoft.sqlserver.jdbc.SQLServerDriver")
        .option("sessionInitStatement", batch_sql)
        .option("dbtable", "(SELECT 1 AS ok) t")
        .load()
        .collect()
    )

print("✅ Função _execute_sql definida (via sessionInitStatement — compatível com Serverless).")


def save_gold_sqlserver_merge(df, table_name, pk_cols):
    """
    Upsert no SQL Server via staging + MERGE T-SQL
    """
    pk_list = [pk_cols] if isinstance(pk_cols, str) else pk_cols
    stg_name = f"squad3.stg_{table_name}"
    tgt_name = f"squad3.{table_name}"

    print(f"\n{'─'*60}")
    print(f"🔄 SQL Server MERGE: {tgt_name}")
    print(f"   Staging: {stg_name}")
    print(f"{'─'*60}")

    # Etapa 1: gravar staging via Spark
    (
        df.write.format("sqlserver")
        .options(**sqlserver_opts)
        .option("dbtable", stg_name)
        .mode("overwrite")
        .save()
    )
    print(f"   ✅ Staging gravado: {df.count()} registros")

    # Etapa 2: construir MERGE T-SQL
    cols = df.columns
    on_clause      = " AND ".join([f"T.[{c}] = S.[{c}]" for c in pk_list])
    update_clause  = ", ".join([f"T.[{c}] = S.[{c}]" for c in cols if c not in pk_list])
    insert_cols    = ", ".join([f"[{c}]" for c in cols])
    insert_vals    = ", ".join([f"S.[{c}]" for c in cols])

    merge_sql = f"""
    MERGE {tgt_name} AS T
    USING {stg_name} AS S
    ON {on_clause}
    WHEN MATCHED THEN UPDATE SET {update_clause}
    WHEN NOT MATCHED BY TARGET THEN
        INSERT ({insert_cols}) VALUES ({insert_vals});
    """
    drop_sql = f"DROP TABLE {stg_name};"

    # Etapa 3: executar via JDBC
    _execute_sql([merge_sql, drop_sql])
    print(f"   ✅ MERGE executado + staging removido")

print("✅ Função save_gold_sqlserver_merge definida.")


def save_gold_sqlserver_overwrite(df, table_name):
    """
    TRUNCATE + INSERT no SQL Server (JDBC, compatível com Serverless).
    """
    tgt_name = f"squad3.{table_name}"

    print(f"\n{'─'*60}")
    print(f"🔄 SQL Server OVERWRITE (TRUNCATE+INSERT): {tgt_name}")
    print(f"{'─'*60}")

    _execute_sql(f"TRUNCATE TABLE {tgt_name};")
    print(f"   ✅ TRUNCATE executado")

    (
        df.write.format("sqlserver")
        .options(**sqlserver_opts)
        .option("dbtable", tgt_name)
        .mode("append")
        .save()
    )
    print(f"   ✅ Insert concluído: {df.count()} registros")

print("✅ Função save_gold_sqlserver_overwrite definida.")

In [0]:
# ── Smoke test: valida save_gold_delta_merge com DataFrame descartável ──
# Cria uma tabela temporária, grava, atualiza via merge, verifica e deleta.
from pyspark.sql import Row
import time

TEST_PATH = f"{GOLD_BASE}/gold_test_delete_me"

# Teste 1: escrita inicial (3 registros)
df_test_1 = spark.createDataFrame([
    Row(id=1, valor="A", ano=2024),
    Row(id=2, valor="B", ano=2024),
    Row(id=3, valor="C", ano=2025),
]).withColumn("gold_processed_at", current_timestamp())

print("── Teste 1: escrita inicial ──")
save_gold_delta_merge(df_test_1, TEST_PATH, pk_cols="id", table_name="gold_test_delete_me")

# Teste 2: merge (atualiza id=2, insere id=4)
time.sleep(1)  # garante timestamp diferente para a dedup funcionar
df_test_2 = spark.createDataFrame([
    Row(id=2, valor="B_ATUALIZADO", ano=2024),
    Row(id=4, valor="D", ano=2025),
]).withColumn("gold_processed_at", current_timestamp())

print("\n── Teste 2: merge (upsert) ──")
save_gold_delta_merge(df_test_2, TEST_PATH, pk_cols="id", table_name="gold_test_delete_me")

# Verificação: deve ter 4 registros, id=2 com 'B_ATUALIZADO'
print("\n── Verificação ──")
df_check = spark.read.format("delta").options(**adls_options).load(TEST_PATH)
df_check.orderBy("id").show(truncate=False)

assert df_check.count() == 4, f"Esperado 4 registros, veio {df_check.count()}"
valor_id2 = df_check.filter(col("id") == 2).collect()[0]["valor"]
assert valor_id2 == "B_ATUALIZADO", f"Esperado 'B_ATUALIZADO', veio '{valor_id2}'"
print("✅ Asserções OK — save_gold_delta_merge funciona corretamente")

# Limpeza
try:
    dbutils.fs.rm(TEST_PATH, recurse=True)
    print(f"🧹 Tabela de teste removida: {TEST_PATH}")
except Exception as e:
    print(f"⚠️  Limpeza falhou (limitação do Serverless): {str(e)[:80]}")
    print(f"   Tabela {TEST_PATH} permanece — remover manualmente se desejar.")
    print(f"   Isso NÃO afeta o resultado do smoke test (asserções passaram acima).")

---
## Fim do Sprint 1

Pronto para o Sprint 2: construção das 4 tabelas Gold de fornecedores (F6, F7+F8, F9, F10).

Status desta rodada:
- ✅ Setup carregado (ADLS + SQL Server)
- ✅ Silvers lidas (principais + referência com fallback Raw)
- ✅ Enriquecimento base (`df_lotes_enriq`, `df_fornecedores_limpos`)
- ✅ 4 funções de gravação definidas
- ✅ Teste de fumaça: save_gold_delta_merge OK

In [0]:
# ── Imports do BLOCO 5 — construção das tabelas Gold ────────────────────
from pyspark.sql.functions import (
    avg, count, countDistinct, create_map, datediff, lit,
    min as spark_min, quarter, round as spark_round, sum as spark_sum,
    when, year, broadcast
)
from itertools import chain

In [0]:
# ══════════════════════════════════════════════════════════════════════════
# F6 — Lead time médio por tipo_fornecedor por trimestre
# Grão: (tipo_fornecedor, ano, trimestre)
# Interpretação: ano/trimestre vêm da dt_fabricacao do lote (snapshot dos
# fornecedores produzindo no período, não dos cadastrados).
# ══════════════════════════════════════════════════════════════════════════

print("\n" + "=" * 60)
print("🏗️  Construindo: gold_fornecedor_lead_time_trimestre")
print("=" * 60)

df_lead_time_tri = (
    df_lotes
    .filter(~col("_flag_fornecedor_orfao"))
    .join(
        broadcast(
            df_fornecedores_limpos
            .select("id_fornecedor", "tipo_fornecedor", "lead_time_dias")
        ),
        "id_fornecedor",
        "inner"
    )
    .withColumn("ano", year("dt_fabricacao"))
    .withColumn("trimestre", quarter("dt_fabricacao"))
    .groupBy("tipo_fornecedor", "ano", "trimestre")
    .agg(
        spark_round(avg("lead_time_dias"), 2).alias("lead_time_medio"),
        countDistinct("id_fornecedor").alias("qtd_forn_produzindo"),
        countDistinct("id_lote").alias("qtd_lotes"),
    )
    .withColumn("gold_processed_at", current_timestamp())
)

# Gravar na Gold Delta
save_gold_delta_merge(
    df_new=df_lead_time_tri,
    gold_path=f"{GOLD_BASE}/gold_fornecedor_lead_time_trimestre",
    pk_cols=["tipo_fornecedor", "ano", "trimestre"],
    table_name="gold_fornecedor_lead_time_trimestre",
)

# Conferência visual
print("\n── Prévia ──")
df_lead_time_tri.orderBy("ano", "trimestre", "tipo_fornecedor").show(20, truncate=False)

In [0]:
# ══════════════════════════════════════════════════════════════════════════
# F7 + F8 — % de fornecedores ativos por categoria + alerta sem ativo
# Grão: categoria
# Base: left join a partir das categorias de referência (ecommerce + perecíveis)
#       para pegar categorias com ZERO fornecedores (regra 8).
# ══════════════════════════════════════════════════════════════════════════

print("\n" + "=" * 60)
print("🏗️  Construindo: gold_fornecedor_ativos_categoria")
print("=" * 60)

# Lista de categorias válidas (duas fontes)
categorias_ecom = (
    df_ecom_categ
    .filter(col("tipo_categoria") == "Categoria")
    .select(col("nome_categoria").alias("categoria"))
    .distinct()
)
categorias_perec = (
    df_pereciveis
    .select(col("categoria_pai").alias("categoria"))
    .distinct()
)
categorias_ref = categorias_ecom.union(categorias_perec).distinct()

print(f"   📂 Categorias de referência: {categorias_ref.count()}")

# Agregações dos fornecedores por categoria
forn_por_categoria = (
    df_fornecedores
    .filter(~col("_flag_categoria_invalida"))
    .groupBy(col("categoria_fornecida").alias("categoria"))
    .agg(
        count("*").alias("qtd_total"),
        spark_sum(when(col("is_ativo"), 1).otherwise(0)).alias("qtd_ativos"),
    )
)

# Left join + flags
df_ativos_cat = (
    categorias_ref.alias("c")
    .join(forn_por_categoria.alias("f"), "categoria", "left")
    .select(
        col("categoria"),
        when(col("qtd_total").isNull(), lit(0)).otherwise(col("qtd_total")).alias("qtd_total"),
        when(col("qtd_ativos").isNull(), lit(0)).otherwise(col("qtd_ativos")).alias("qtd_ativos"),
    )
    .withColumn(
        "pct_ativos",
        when(col("qtd_total") > 0,
             spark_round(col("qtd_ativos") / col("qtd_total") * 100, 2))
        .otherwise(lit(0.0))
    )
    .withColumn("flag_alerta_sem_ativo", col("qtd_ativos") == 0)
    .withColumn("flag_risco_concentracao",
                (col("qtd_ativos") > 0) & (col("qtd_ativos") <= 2))
    .withColumn("gold_processed_at", current_timestamp())
)

save_gold_delta_merge(
    df_new=df_ativos_cat,
    gold_path=f"{GOLD_BASE}/gold_fornecedor_ativos_categoria",
    pk_cols="categoria",
    table_name="gold_fornecedor_ativos_categoria",
)

# Conferência: alertas primeiro (categorias sem fornecedor ativo)
print("\n── Prévia (alertas no topo) ──")
df_ativos_cat.orderBy(
    col("flag_alerta_sem_ativo").desc(),
    col("flag_risco_concentracao").desc(),
    col("qtd_ativos").asc()
).show(30, truncate=False)

# Resumo dos alertas
qtd_sem_ativo = df_ativos_cat.filter(col("flag_alerta_sem_ativo")).count()
qtd_risco     = df_ativos_cat.filter(col("flag_risco_concentracao")).count()
print(f"\n⚠️  Categorias SEM nenhum fornecedor ativo:  {qtd_sem_ativo}")
print(f"⚠️  Categorias com ≤ 2 ativos (concentração): {qtd_risco}")

In [0]:
# ══════════════════════════════════════════════════════════════════════════
# F9 — Distribuição de fornecedores por uf_origem
# Grão: uf_origem
# Enriquecimento: coluna 'regiao' do Brasil (Sul/Sudeste/etc) para
# facilitar filtros macro-regionais no Looker.
# ══════════════════════════════════════════════════════════════════════════

print("\n" + "=" * 60)
print("🏗️  Construindo: gold_fornecedor_distribuicao_uf")
print("=" * 60)

# Mapa UF → Região (fonte: IBGE)
mapa_regiao = {
    "AC": "Norte",        "AP": "Norte",        "AM": "Norte",        "PA": "Norte",
    "RO": "Norte",        "RR": "Norte",        "TO": "Norte",
    "AL": "Nordeste",     "BA": "Nordeste",     "CE": "Nordeste",     "MA": "Nordeste",
    "PB": "Nordeste",     "PE": "Nordeste",     "PI": "Nordeste",     "RN": "Nordeste",
    "SE": "Nordeste",
    "DF": "Centro-Oeste", "GO": "Centro-Oeste", "MT": "Centro-Oeste", "MS": "Centro-Oeste",
    "ES": "Sudeste",      "MG": "Sudeste",      "RJ": "Sudeste",      "SP": "Sudeste",
    "PR": "Sul",          "RS": "Sul",          "SC": "Sul",
}

# Expressão Spark para o lookup
mapa_regiao_expr = create_map([lit(x) for x in chain(*mapa_regiao.items())])

# Universo total para calcular pct_total
total_fornecedores = df_fornecedores.filter(~col("_flag_uf_invalida")).count()
print(f"   📂 Universo total: {total_fornecedores} fornecedores com UF válida")

df_distribuicao_uf = (
    df_fornecedores
    .filter(~col("_flag_uf_invalida"))
    .groupBy("uf_origem")
    .agg(
        count("*").alias("qtd_fornecedores"),
        spark_sum(when(col("is_ativo"), 1).otherwise(0)).alias("qtd_ativos"),
    )
    .withColumn(
        "pct_total",
        spark_round(col("qtd_fornecedores") / total_fornecedores * 100, 2)
    )
    .withColumn("regiao", mapa_regiao_expr[col("uf_origem")])
    .withColumn("gold_processed_at", current_timestamp())
)

save_gold_delta_merge(
    df_new=df_distribuicao_uf,
    gold_path=f"{GOLD_BASE}/gold_fornecedor_distribuicao_uf",
    pk_cols="uf_origem",
    table_name="gold_fornecedor_distribuicao_uf",
)

# Conferência
print("\n── Prévia (top UFs) ──")
df_distribuicao_uf.orderBy(col("qtd_fornecedores").desc()).show(30, truncate=False)

print("\n── Agregado por região ──")
df_distribuicao_uf.groupBy("regiao").agg(
    spark_sum("qtd_fornecedores").alias("total_forn"),
    spark_sum("qtd_ativos").alias("total_ativos"),
    count("*").alias("qtd_ufs"),
).orderBy(col("total_forn").desc()).show(truncate=False)

In [0]:
# ══════════════════════════════════════════════════════════════════════════
# F10 — Tempo entre dt_cadastro e primeira dt_fabricacao de um lote
# Grão: id_fornecedor (um registro por fornecedor)
# Da exploração: todos os 60 têm lote; zero negativos. Left join defensivo.
# ══════════════════════════════════════════════════════════════════════════

print("\n" + "=" * 60)
print("🏗️  Construindo: gold_fornecedor_tempo_ativacao")
print("=" * 60)

# Primeira dt_fabricacao por fornecedor
primeiro_lote = (
    df_lotes
    .filter(~col("_flag_fornecedor_orfao"))
    .groupBy("id_fornecedor")
    .agg(spark_min("dt_fabricacao").alias("dt_primeiro_lote"))
)

df_tempo_ativacao = (
    df_fornecedores
    .select("id_fornecedor", "razao_social", "tipo_fornecedor", "dt_cadastro")
    .join(primeiro_lote, "id_fornecedor", "left")
    .withColumn(
        "dias_ate_ativacao",
        datediff(col("dt_primeiro_lote"), col("dt_cadastro"))
    )
    .withColumn("gold_processed_at", current_timestamp())
)

save_gold_delta_merge(
    df_new=df_tempo_ativacao,
    gold_path=f"{GOLD_BASE}/gold_fornecedor_tempo_ativacao",
    pk_cols="id_fornecedor",
    table_name="gold_fornecedor_tempo_ativacao",
)

# Conferência: top 10 mais lentos e top 10 mais rápidos
print("\n── 10 ativações mais LENTAS ──")
df_tempo_ativacao.orderBy(col("dias_ate_ativacao").desc()).show(10, truncate=False)

print("── 10 ativações mais RÁPIDAS ──")
df_tempo_ativacao.orderBy(col("dias_ate_ativacao").asc()).show(10, truncate=False)

# Checagem defensiva: nenhum NULL (todos os forn devem ter lote)
qtd_sem_lote = df_tempo_ativacao.filter(col("dt_primeiro_lote").isNull()).count()
if qtd_sem_lote > 0:
    print(f"\n⚠️  {qtd_sem_lote} fornecedor(es) sem lote — inesperado, investigar")
else:
    print("\n✅ Todos os fornecedores têm pelo menos um lote (consistente com exploração)")

In [0]:
# ══════════════════════════════════════════════════════════════════════════
# L6 5.5 — Alerta: lotes 'Ativo' com dt_validade vencida
# Grão: id_lote (granular, acionável no Looker)
# Modo: OVERWRITE (snapshot — lote baixado do ERP deve SAIR do alerta)
# Severidade: critico > 180d, alto > 30d, moderado <= 30d
# ══════════════════════════════════════════════════════════════════════════

print("\n" + "=" * 60)
print("🏗️  Construindo: gold_lote_alerta_ativo_vencido")
print("=" * 60)

df_alerta_l6 = (
    df_lotes_enriq
    .filter(~col("_flag_status_invalido") & ~col("_flag_validade_invertida"))
    .filter(col("status") == "Ativo")
    .filter(col("dt_validade") < DATA_REFERENCIA)
    .withColumn("data_referencia", DATA_REFERENCIA)
    .withColumn("dias_vencido", datediff(col("data_referencia"), col("dt_validade")))
    .withColumn(
        "severidade",
        when(col("dias_vencido") > 180, lit("critico"))
        .when(col("dias_vencido") > 30, lit("alto"))
        .otherwise(lit("moderado"))
    )
    .withColumn("ano", year("dt_validade"))
    .select(
        "id_lote", "sku", "id_fornecedor", "razao_social",
        "dt_fabricacao", "dt_validade", "data_referencia", "dias_vencido",
        "temperatura_armazenamento_ideal", "categoria_pai", "severidade",
        "ano"
    )
    .withColumn("gold_processed_at", current_timestamp())
)

save_gold_delta_overwrite(
    df=df_alerta_l6,
    gold_path=f"{GOLD_BASE}/gold_lote_alerta_ativo_vencido",
    table_name="gold_lote_alerta_ativo_vencido",
)

# Conferência
qtd = df_alerta_l6.count()
print(f"\n⚠️  Total de lotes em alerta: {qtd}")

print("\n── Distribuição por severidade ──")
df_alerta_l6.groupBy("severidade").agg(
    count("*").alias("qtd_lotes"),
    spark_round(avg("dias_vencido"), 1).alias("media_dias_vencido"),
    spark_min("dias_vencido").alias("min_dias"),
).orderBy(
    when(col("severidade") == "critico", 1)
    .when(col("severidade") == "alto", 2)
    .otherwise(3)
).show(truncate=False)

print("── Top 10 lotes mais vencidos ──")
df_alerta_l6.orderBy(col("dias_vencido").desc()) \
    .select("id_lote", "razao_social", "sku", "dt_validade", "dias_vencido", "severidade") \
    .show(10, truncate=False)

In [0]:
# ══════════════════════════════════════════════════════════════════════════
# L7 5.6 — Volume produzido por categoria por mês
# Grão: (canal, categoria, ano, mes)
# Modelo híbrido:
#   - Perecíveis (70%)  → categoria_pai real
#   - E-commerce (30%)  → agregado como 'E-commerce (não perecível)'
# ══════════════════════════════════════════════════════════════════════════

print("\n" + "=" * 60)
print("🏗️  Construindo: gold_lote_volume_categoria_mes")
print("=" * 60)

from pyspark.sql.functions import month as spark_month

df_volume_cat_mes = (
    df_lotes_enriq
    .filter(~col("_flag_quantidade_invalida"))
    .withColumn(
        "canal",
        when(col("categoria_pai").isNotNull(), lit("perecivel"))
        .otherwise(lit("ecommerce"))
    )
    .withColumn(
        "categoria",
        when(col("categoria_pai").isNotNull(), col("categoria_pai"))
        .otherwise(lit("E-commerce (não perecível)"))
    )
    .withColumn("ano", year("dt_fabricacao"))
    .withColumn("mes", spark_month("dt_fabricacao"))
    .groupBy("canal", "categoria", "ano", "mes")
    .agg(
        countDistinct("id_lote").alias("qtd_lotes"),
        spark_sum("quantidade_produzida").alias("volume_total"),
        spark_round(avg("quantidade_produzida"), 2).alias("volume_medio_lote"),
    )
    .withColumn("gold_processed_at", current_timestamp())
)

save_gold_delta_merge(
    df_new=df_volume_cat_mes,
    gold_path=f"{GOLD_BASE}/gold_lote_volume_categoria_mes",
    pk_cols=["canal", "categoria", "ano", "mes"],
    table_name="gold_lote_volume_categoria_mes",
)

# Conferência
print("\n── Volume total por canal ──")
df_volume_cat_mes.groupBy("canal").agg(
    spark_sum("qtd_lotes").alias("total_lotes"),
    spark_sum("volume_total").alias("volume_total"),
).show(truncate=False)

print("── Top 10 (categoria × mês) por volume ──")
df_volume_cat_mes.orderBy(col("volume_total").desc()) \
    .select("canal", "categoria", "ano", "mes", "qtd_lotes", "volume_total", "volume_medio_lote") \
    .show(10, truncate=False)

In [0]:
# ══════════════════════════════════════════════════════════════════════════
# L8 5.7 — % de lotes em 'Recall' por fornecedor por trimestre
# Grão: (id_fornecedor, ano, trimestre)
# Métrica principal de qualidade do fornecedor.
# Coluna significancia_estatistica evita falso positivo em amostras pequenas
# (ex: fornecedor com 1 lote e 1 recall = 100% é estatisticamente irrelevante).
# ══════════════════════════════════════════════════════════════════════════

print("\n" + "=" * 60)
print("🏗️  Construindo: gold_lote_recall_fornecedor_trimestre")
print("=" * 60)

df_recall_tri = (
    df_lotes_enriq
    .filter(~col("_flag_status_invalido") & ~col("_flag_fornecedor_orfao"))
    .withColumn("ano", year("dt_fabricacao"))
    .withColumn("trimestre", quarter("dt_fabricacao"))
    .groupBy("id_fornecedor", "razao_social", "ano", "trimestre")
    .agg(
        count("*").alias("qtd_lotes"),
        spark_sum(when(col("status") == "Recall", 1).otherwise(0)).alias("qtd_recall"),
    )
    .withColumn(
        "pct_recall",
        when(col("qtd_lotes") > 0,
             spark_round(col("qtd_recall") / col("qtd_lotes") * 100, 2))
        .otherwise(lit(0.0))
    )
    .withColumn(
        "significancia_estatistica",
        when(col("qtd_lotes") < 3, lit("baixa_amostra"))
        .when(col("qtd_lotes") < 10, lit("amostra_media"))
        .otherwise(lit("amostra_boa"))
    )
    .withColumn("gold_processed_at", current_timestamp())
)

save_gold_delta_merge(
    df_new=df_recall_tri,
    gold_path=f"{GOLD_BASE}/gold_lote_recall_fornecedor_trimestre",
    pk_cols=["id_fornecedor", "ano", "trimestre"],
    table_name="gold_lote_recall_fornecedor_trimestre",
)

# ── Conferência ──────────────────────────────────────────────────────────
print("\n── Distribuição por significância estatística ──")
df_recall_tri.groupBy("significancia_estatistica").agg(
    count("*").alias("qtd_combinacoes"),
    spark_sum("qtd_lotes").alias("total_lotes"),
    spark_sum("qtd_recall").alias("total_recalls"),
).orderBy(
    when(col("significancia_estatistica") == "amostra_boa", 1)
    .when(col("significancia_estatistica") == "amostra_media", 2)
    .otherwise(3)
).show(truncate=False)

print("── Top 10 piores — SOMENTE amostras confiáveis (média + boa) ──")
df_recall_tri.filter(
    (col("qtd_recall") > 0) & (col("significancia_estatistica") != "baixa_amostra")
) \
    .orderBy(col("pct_recall").desc(), col("qtd_recall").desc()) \
    .show(10, truncate=False)

print("── Top 10 piores — BAIXA amostra (contexto, não acionável) ──")
df_recall_tri.filter(
    (col("qtd_recall") > 0) & (col("significancia_estatistica") == "baixa_amostra")
) \
    .orderBy(col("pct_recall").desc(), col("qtd_recall").desc()) \
    .show(10, truncate=False)

qtd_com_recall = df_recall_tri.filter(col("qtd_recall") > 0).count()
qtd_sem_recall = df_recall_tri.filter(col("qtd_recall") == 0).count()
print(f"\n   Combinações com recall:  {qtd_com_recall}")
print(f"   Combinações sem recall:  {qtd_sem_recall}")

In [0]:
# ══════════════════════════════════════════════════════════════════════════
# L9 5.8 — Tempo médio de prateleira por categoria_pai (perecíveis)
# Grão: categoria_pai
# Compara shelf life observado com janela esperada do negócio
# (janelas vêm da exploração seção 5.1 — confirmar com planilha do mentor)
# ══════════════════════════════════════════════════════════════════════════

print("\n" + "=" * 60)
print("🏗️  Construindo: gold_lote_shelf_life_categoria")
print("=" * 60)

from pyspark.sql.functions import expr

# Janelas esperadas (empíricas, observadas na exploração)
janelas_esperadas = {
    "Padaria Fresca":     (1, 3),
    "Açougue & Peixaria": (3, 8),
    "Hortifruti":         (3, 10),
    "Laticínios & Ovos":  (15, 35),
    "Frios & Embutidos":  (20, 45),
    "Congelados":         (120, 365),
}

# Construir expressões CASE para janela min/max
case_min = "CASE categoria_pai\n" + "\n".join(
    [f"  WHEN '{k}' THEN {v[0]}" for k, v in janelas_esperadas.items()]
) + "\n  ELSE NULL END"
case_max = "CASE categoria_pai\n" + "\n".join(
    [f"  WHEN '{k}' THEN {v[1]}" for k, v in janelas_esperadas.items()]
) + "\n  ELSE NULL END"

df_shelf = (
    df_lotes_enriq
    .filter(~col("_flag_validade_invertida") & ~col("_flag_sku_sem_catalogo"))
    .filter(col("categoria_pai").isNotNull())  # apenas perecíveis
    .withColumn("shelf_life_dias", datediff(col("dt_validade"), col("dt_fabricacao")))
    .groupBy("categoria_pai")
    .agg(
        spark_min("shelf_life_dias").alias("shelf_life_min"),
        expr("percentile(shelf_life_dias, 0.25)").alias("shelf_life_p25"),
        expr("percentile(shelf_life_dias, 0.50)").alias("shelf_life_mediana"),
        spark_round(avg("shelf_life_dias"), 2).alias("shelf_life_media"),
        expr("percentile(shelf_life_dias, 0.75)").alias("shelf_life_p75"),
        expr("max(shelf_life_dias)").alias("shelf_life_max"),
        countDistinct("id_lote").alias("qtd_lotes"),
    )
    .withColumn("janela_esperada_min", expr(case_min))
    .withColumn("janela_esperada_max", expr(case_max))
    .withColumn("gold_processed_at", current_timestamp())
)

save_gold_delta_merge(
    df_new=df_shelf,
    gold_path=f"{GOLD_BASE}/gold_lote_shelf_life_categoria",
    pk_cols="categoria_pai",
    table_name="gold_lote_shelf_life_categoria",
)

# Conferência
print("\n── Shelf life observado vs janela esperada ──")
df_shelf.orderBy("shelf_life_media").show(truncate=False)

In [0]:
# ══════════════════════════════════════════════════════════════════════════
# L10 (fato) 5.9 — Lotes produzidos que NUNCA chegaram em estoque
# Grão: id_lote (granular, acionável)
# Modo: OVERWRITE (lote pode entrar/sair do universo entre execuções)
# ══════════════════════════════════════════════════════════════════════════

print("\n" + "=" * 60)
print("🏗️  Construindo: gold_lote_sem_estoque")
print("=" * 60)

# Confirmar que estoque tem id_lote (checagem defensiva)
if "id_lote" not in df_estoque.columns:
    raise ValueError(
        f"food_estoque_lojas sem coluna 'id_lote'. Colunas: {df_estoque.columns}"
    )

# Lotes que aparecem em pelo menos um snapshot de estoque
lotes_em_estoque = df_estoque.select(col("id_lote")).distinct()

# Lotes produzidos que NUNCA aparecem em estoque (left anti join)
df_sem_estoque = (
    df_lotes_enriq
    .join(lotes_em_estoque, "id_lote", "left_anti")
    .withColumn("ano", year("dt_fabricacao"))
    .select(
        "id_lote", "sku", "id_fornecedor", "razao_social",
        "dt_fabricacao", "status", "categoria_pai", "temperatura_armazenamento_ideal",
        "ano"
    )
    .withColumn("gold_processed_at", current_timestamp())
)

save_gold_delta_overwrite(
    df=df_sem_estoque,
    gold_path=f"{GOLD_BASE}/gold_lote_sem_estoque",
    table_name="gold_lote_sem_estoque",
)

# Conferência
qtd = df_sem_estoque.count()
total = df_lotes.count()
print(f"\n   Lotes sem estoque: {qtd}/{total} ({qtd/total*100:.1f}%)")

print("\n── Distribuição por status ──")
df_sem_estoque.groupBy("status").count().orderBy(col("count").desc()).show()

print("── Top 10 fornecedores com mais lotes não entregues ──")
df_sem_estoque.groupBy("id_fornecedor", "razao_social").count() \
    .orderBy(col("count").desc()).show(10, truncate=False)

In [0]:
# ══════════════════════════════════════════════════════════════════════════
# L10 (dimensão) 5.10 — Perfil de distribuição por fornecedor
# Grão: id_fornecedor
# Classifica fornecedores em: loja_fisica / misto / atacado_b2b
#   taxa >= 80%  → loja_fisica
#   20-80%       → misto
#   < 20%        → atacado_b2b (produz mas não abastece lojas)
# ══════════════════════════════════════════════════════════════════════════

print("\n" + "=" * 60)
print("🏗️  Construindo: gold_fornecedor_canal_distribuicao")
print("=" * 60)

# Lotes por fornecedor (universo total)
lotes_por_forn = (
    df_lotes_enriq
    .groupBy("id_fornecedor", "razao_social")
    .agg(countDistinct("id_lote").alias("qtd_lotes_total"))
)

# Snapshots de estoque: lotes + loja + dias até primeiro snapshot
estoque_enriq = (
    df_estoque
    .select(
        col("id_lote"),
        col("id_loja"),
        col("dt_snapshot")
    )
    .join(
        df_lotes_enriq.select("id_lote", "id_fornecedor", "dt_fabricacao"),
        "id_lote",
        "inner"
    )
    .withColumn(
        "dias_fab_ate_snapshot",
        datediff(col("dt_snapshot"), col("dt_fabricacao"))
    )
    .filter(col("dias_fab_ate_snapshot") >= 0)  # ignora snapshots inválidos (antes da fab)
)

# Agregações por fornecedor sobre o estoque enriquecido
estoque_agg = (
    estoque_enriq
    .groupBy("id_fornecedor")
    .agg(
        countDistinct("id_lote").alias("qtd_lotes_com_estoque"),
        countDistinct("id_loja").alias("qtd_lojas_distintas_atingidas"),
        spark_round(avg("dias_fab_ate_snapshot"), 1).alias("media_dias_ate_primeiro_snapshot"),
    )
)

# Junta universo total + agregações; calcula taxa + perfil
df_canal = (
    lotes_por_forn
    .join(estoque_agg, "id_fornecedor", "left")
    .withColumn(
        "qtd_lotes_com_estoque",
        when(col("qtd_lotes_com_estoque").isNull(), lit(0))
        .otherwise(col("qtd_lotes_com_estoque"))
    )
    .withColumn(
        "qtd_lojas_distintas_atingidas",
        when(col("qtd_lojas_distintas_atingidas").isNull(), lit(0))
        .otherwise(col("qtd_lojas_distintas_atingidas"))
    )
    .withColumn(
        "qtd_lotes_sem_estoque",
        col("qtd_lotes_total") - col("qtd_lotes_com_estoque")
    )
    .withColumn(
        "taxa_chegada_loja",
        spark_round(col("qtd_lotes_com_estoque") / col("qtd_lotes_total") * 100, 2)
    )
    .withColumn(
        "perfil",
        when(col("taxa_chegada_loja") >= 80, lit("loja_fisica"))
        .when(col("taxa_chegada_loja") >= 20, lit("misto"))
        .otherwise(lit("atacado_b2b"))
    )
    .select(
        "id_fornecedor", "razao_social",
        "qtd_lotes_total", "qtd_lotes_com_estoque", "qtd_lotes_sem_estoque",
        "taxa_chegada_loja",
        "qtd_lojas_distintas_atingidas", "media_dias_ate_primeiro_snapshot",
        "perfil",
    )
    .withColumn("gold_processed_at", current_timestamp())
)

save_gold_delta_merge(
    df_new=df_canal,
    gold_path=f"{GOLD_BASE}/gold_fornecedor_canal_distribuicao",
    pk_cols="id_fornecedor",
    table_name="gold_fornecedor_canal_distribuicao",
)

# Conferência
print("\n── Distribuição de perfis ──")
df_canal.groupBy("perfil").agg(
    count("*").alias("qtd_fornecedores"),
    spark_round(avg("taxa_chegada_loja"), 1).alias("media_taxa_chegada"),
    spark_round(avg("qtd_lojas_distintas_atingidas"), 1).alias("media_lojas_atingidas"),
).orderBy("qtd_fornecedores", ascending=False).show(truncate=False)

print("── 10 fornecedores de perfil ATACADO_B2B (candidatos à hipótese) ──")
df_canal.filter(col("perfil") == "atacado_b2b") \
    .orderBy(col("taxa_chegada_loja").asc()) \
    .select("id_fornecedor", "razao_social", "qtd_lotes_total", "qtd_lotes_sem_estoque",
            "taxa_chegada_loja", "qtd_lojas_distintas_atingidas", "perfil") \
    .show(10, truncate=False)

print("── 10 fornecedores de perfil LOJA_FISICA (melhor cobertura) ──")
df_canal.filter(col("perfil") == "loja_fisica") \
    .orderBy(col("qtd_lojas_distintas_atingidas").desc()) \
    .select("id_fornecedor", "razao_social", "qtd_lotes_total", "taxa_chegada_loja",
            "qtd_lojas_distintas_atingidas", "media_dias_ate_primeiro_snapshot") \
    .show(10, truncate=False)

In [0]:
# ══════════════════════════════════════════════════════════════════════════
# BLOCO 6 — DDL do SQL Server (schema squad3 + 10 tabelas Gold)
# Idempotente: usa IF OBJECT_ID(...) IS NULL para não falhar em re-execução.
# Tipos escolhidos para otimizar Looker: decimal(10,2) em percentuais,
# datetime2 em timestamps, nvarchar com tamanhos realistas.
# ══════════════════════════════════════════════════════════════════════════

from pyspark.sql.functions import row_number, current_timestamp
from pyspark.sql.window import Window

print("\n" + "=" * 60)
print("🏗️  BLOCO 6 — DDL do SQL Server")
print("=" * 60)

ddl_statements = [

    # ── Schema ─────────────────────────────────────────────────────────
    """
    IF NOT EXISTS (SELECT 1 FROM sys.schemas WHERE name = 'squad3')
        EXEC('CREATE SCHEMA squad3');
    """,

    # ── 1. gold_fornecedor_lead_time_trimestre (F6) ────────────────────
    """
    IF OBJECT_ID('squad3.gold_fornecedor_lead_time_trimestre', 'U') IS NULL
    CREATE TABLE squad3.gold_fornecedor_lead_time_trimestre (
        tipo_fornecedor      nvarchar(50)   NOT NULL,
        ano                  int            NOT NULL,
        trimestre            int            NOT NULL,
        lead_time_medio      decimal(10,2),
        qtd_forn_produzindo  int,
        qtd_lotes            int,
        gold_processed_at    datetime2,
        CONSTRAINT pk_gold_fornecedor_lead_time_trimestre
            PRIMARY KEY (tipo_fornecedor, ano, trimestre)
    );
    """,

    # ── 2. gold_fornecedor_ativos_categoria (F7+F8) ────────────────────
    """
    IF OBJECT_ID('squad3.gold_fornecedor_ativos_categoria', 'U') IS NULL
    CREATE TABLE squad3.gold_fornecedor_ativos_categoria (
        categoria                 nvarchar(100)  NOT NULL,
        qtd_total                 int,
        qtd_ativos                int,
        pct_ativos                decimal(10,2),
        flag_alerta_sem_ativo     bit,
        flag_risco_concentracao   bit,
        gold_processed_at         datetime2,
        CONSTRAINT pk_gold_fornecedor_ativos_categoria
            PRIMARY KEY (categoria)
    );
    """,

    # ── 3. gold_fornecedor_distribuicao_uf (F9) ────────────────────────
    """
    IF OBJECT_ID('squad3.gold_fornecedor_distribuicao_uf', 'U') IS NULL
    CREATE TABLE squad3.gold_fornecedor_distribuicao_uf (
        uf_origem         nvarchar(2)    NOT NULL,
        qtd_fornecedores  int,
        qtd_ativos        int,
        pct_total         decimal(10,2),
        regiao            nvarchar(20),
        gold_processed_at datetime2,
        CONSTRAINT pk_gold_fornecedor_distribuicao_uf
            PRIMARY KEY (uf_origem)
    );
    """,

    # ── 4. gold_fornecedor_tempo_ativacao (F10) ────────────────────────
    """
    IF OBJECT_ID('squad3.gold_fornecedor_tempo_ativacao', 'U') IS NULL
    CREATE TABLE squad3.gold_fornecedor_tempo_ativacao (
        id_fornecedor     int            NOT NULL,
        razao_social      nvarchar(200),
        tipo_fornecedor   nvarchar(50),
        dt_cadastro       datetime2,
        dt_primeiro_lote  datetime2,
        dias_ate_ativacao int,
        gold_processed_at datetime2,
        CONSTRAINT pk_gold_fornecedor_tempo_ativacao
            PRIMARY KEY (id_fornecedor)
    );
    """,

    # ── 5. gold_lote_alerta_ativo_vencido (L6) ─────────────────────────
    """
    IF OBJECT_ID('squad3.gold_lote_alerta_ativo_vencido', 'U') IS NULL
    CREATE TABLE squad3.gold_lote_alerta_ativo_vencido (
        id_lote                          int           NOT NULL,
        sku                              nvarchar(200),
        id_fornecedor                    int,
        razao_social                     nvarchar(200),
        dt_fabricacao                    datetime2,
        dt_validade                      datetime2,
        data_referencia                  date,
        dias_vencido                     int,
        temperatura_armazenamento_ideal  nvarchar(20),
        categoria_pai                    nvarchar(100),
        severidade                       nvarchar(10),
        ano                              int,
        gold_processed_at                datetime2,
        CONSTRAINT pk_gold_lote_alerta_ativo_vencido
            PRIMARY KEY (id_lote)
    );
    """,

    # ── 6. gold_lote_volume_categoria_mes (L7) ─────────────────────────
    """
    IF OBJECT_ID('squad3.gold_lote_volume_categoria_mes', 'U') IS NULL
    CREATE TABLE squad3.gold_lote_volume_categoria_mes (
        canal              nvarchar(20)   NOT NULL,
        categoria          nvarchar(100)  NOT NULL,
        ano                int            NOT NULL,
        mes                int            NOT NULL,
        qtd_lotes          int,
        volume_total       bigint,
        volume_medio_lote  decimal(14,2),
        gold_processed_at  datetime2,
        CONSTRAINT pk_gold_lote_volume_categoria_mes
            PRIMARY KEY (canal, categoria, ano, mes)
    );
    """,

    # ── 7. gold_lote_recall_fornecedor_trimestre (L8) ──────────────────
    """
    IF OBJECT_ID('squad3.gold_lote_recall_fornecedor_trimestre', 'U') IS NULL
    CREATE TABLE squad3.gold_lote_recall_fornecedor_trimestre (
        id_fornecedor               int            NOT NULL,
        razao_social                nvarchar(200),
        ano                         int            NOT NULL,
        trimestre                   int            NOT NULL,
        qtd_lotes                   int,
        qtd_recall                  int,
        pct_recall                  decimal(10,2),
        significancia_estatistica   nvarchar(20),
        gold_processed_at           datetime2,
        CONSTRAINT pk_gold_lote_recall_fornecedor_trimestre
            PRIMARY KEY (id_fornecedor, ano, trimestre)
    );
    """,

    # ── 8. gold_lote_shelf_life_categoria (L9) ─────────────────────────
    """
    IF OBJECT_ID('squad3.gold_lote_shelf_life_categoria', 'U') IS NULL
    CREATE TABLE squad3.gold_lote_shelf_life_categoria (
        categoria_pai         nvarchar(100)  NOT NULL,
        shelf_life_min        int,
        shelf_life_p25        decimal(10,2),
        shelf_life_mediana    decimal(10,2),
        shelf_life_media      decimal(10,2),
        shelf_life_p75        decimal(10,2),
        shelf_life_max        int,
        qtd_lotes             int,
        janela_esperada_min   int,
        janela_esperada_max   int,
        gold_processed_at     datetime2,
        CONSTRAINT pk_gold_lote_shelf_life_categoria
            PRIMARY KEY (categoria_pai)
    );
    """,

    # ── 9. gold_lote_sem_estoque (L10 — fato) ──────────────────────────
    """
    IF OBJECT_ID('squad3.gold_lote_sem_estoque', 'U') IS NULL
    CREATE TABLE squad3.gold_lote_sem_estoque (
        id_lote                          int           NOT NULL,
        sku                              nvarchar(200),
        id_fornecedor                    int,
        razao_social                     nvarchar(200),
        dt_fabricacao                    datetime2,
        status                           nvarchar(20),
        categoria_pai                    nvarchar(100),
        temperatura_armazenamento_ideal  nvarchar(20),
        ano                              int,
        gold_processed_at                datetime2,
        CONSTRAINT pk_gold_lote_sem_estoque
            PRIMARY KEY (id_lote)
    );
    """,

    # ── 10. gold_fornecedor_canal_distribuicao (L10 — dimensão) ────────
    """
    IF OBJECT_ID('squad3.gold_fornecedor_canal_distribuicao', 'U') IS NULL
    CREATE TABLE squad3.gold_fornecedor_canal_distribuicao (
        id_fornecedor                     int            NOT NULL,
        razao_social                      nvarchar(200),
        qtd_lotes_total                   int,
        qtd_lotes_com_estoque             int,
        qtd_lotes_sem_estoque             int,
        taxa_chegada_loja                 decimal(10,2),
        qtd_lojas_distintas_atingidas     int,
        media_dias_ate_primeiro_snapshot  decimal(10,2),
        perfil                            nvarchar(20),
        gold_processed_at                 datetime2,
        CONSTRAINT pk_gold_fornecedor_canal_distribuicao
            PRIMARY KEY (id_fornecedor)
    );
    """,
]

# Executar via JDBC (Spark Connect: sessionInitStatement executa DDL sem _jvm)
_jdbc_url = (
    f"jdbc:sqlserver://{SQL_HOST}:1433;"
    f"database={SQL_DB};encrypt=true;trustServerCertificate=false;"
    "hostNameInCertificate=*.database.windows.net;loginTimeout=30;"
)
spark.read.format("jdbc") \
    .option("url", _jdbc_url) \
    .option("user", SQL_USER) \
    .option("password", SQL_PASS) \
    .option("driver", "com.microsoft.sqlserver.jdbc.SQLServerDriver") \
    .option("sessionInitStatement", "\n".join(ddl_statements)) \
    .option("dbtable", "(SELECT 1 AS ok) t") \
    .load().count()

print(f"\n✅ DDL executado: 1 schema + 10 tabelas verificadas/criadas")
print(f"   Todas as tabelas em: {SQL_DB}.squad3.*")
print(f"   Idempotente: pode re-executar sem efeito colateral")

# ── Validação: listar as tabelas criadas ─────────────────────────────────
# Lê do SQL via Spark para confirmar que as tabelas estão lá
print("\n── Verificação: tabelas presentes no schema squad3 ──")

jdbc_url = (
    f"jdbc:sqlserver://{SQL_HOST}:1433;"
    f"database={SQL_DB};encrypt=true;trustServerCertificate=false;"
    "hostNameInCertificate=*.database.windows.net;loginTimeout=30;"
)

df_tables = (
    spark.read.format("jdbc")
    .option("url", jdbc_url)
    .option("user", SQL_USER)
    .option("password", SQL_PASS)
    .option("driver", "com.microsoft.sqlserver.jdbc.SQLServerDriver")
    .option("query", """
        SELECT TABLE_NAME, 
               (SELECT COUNT(*) FROM INFORMATION_SCHEMA.COLUMNS c 
                WHERE c.TABLE_SCHEMA = 'squad3' 
                  AND c.TABLE_NAME = t.TABLE_NAME) AS qtd_colunas
        FROM INFORMATION_SCHEMA.TABLES t
        WHERE t.TABLE_SCHEMA = 'squad3' 
          AND t.TABLE_NAME LIKE 'gold_%'
    """)
    .load()
    .orderBy("TABLE_NAME")
)

df_tables.show(20, truncate=False)
qtd = df_tables.count()
print(f"✅ {qtd} tabelas 'gold_*' no schema squad3")
if qtd != 10:
    print(f"⚠️  Esperado 10 tabelas, encontradas {qtd} — verificar DDL")

## BLOCO 7 — Gravação no SQL Server

Para cada uma das 10 tabelas Gold em Delta:
1. Lê o Delta do ADLS
2. Grava no SQL Server (schema `squad3`) via staging + MERGE T-SQL (ou TRUNCATE+INSERT para alertas)

**Pré-requisitos:** Bloco 6 (DDL) já executado — todas as 10 tabelas devem existir no SQL Server.

### Políticas de gravação
- **MERGE** (upsert por PK): 8 tabelas acumulativas
- **TRUNCATE + INSERT**: 2 tabelas de alerta (L6, L10 fato), onde chaves antigas devem ser removidas entre execuções

In [0]:
# ── 7.1 — gold_fornecedor_lead_time_trimestre (F6) ──────────────────────
tabela = "gold_fornecedor_lead_time_trimestre"
df_delta = spark.read.format("delta").options(**adls_options).load(f"{GOLD_BASE}/{tabela}")
print(f"\n📂 Delta: {df_delta.count()} registros lidos")

save_gold_sqlserver_merge(
    df=df_delta,
    table_name=tabela,
    pk_cols=["tipo_fornecedor", "ano", "trimestre"],
)

In [0]:
# ── 7.2 — gold_fornecedor_ativos_categoria (F7+F8) ──────────────────────
tabela = "gold_fornecedor_ativos_categoria"
df_delta = spark.read.format("delta").options(**adls_options).load(f"{GOLD_BASE}/{tabela}")
print(f"\n📂 Delta: {df_delta.count()} registros lidos")

save_gold_sqlserver_merge(
    df=df_delta,
    table_name=tabela,
    pk_cols="categoria",
)

In [0]:
# ── 7.3 — gold_fornecedor_distribuicao_uf (F9) ──────────────────────────
tabela = "gold_fornecedor_distribuicao_uf"
df_delta = spark.read.format("delta").options(**adls_options).load(f"{GOLD_BASE}/{tabela}")
print(f"\n📂 Delta: {df_delta.count()} registros lidos")

save_gold_sqlserver_merge(
    df=df_delta,
    table_name=tabela,
    pk_cols="uf_origem",
)

In [0]:
# ── 7.4 — gold_fornecedor_tempo_ativacao (F10) ──────────────────────────
tabela = "gold_fornecedor_tempo_ativacao"
df_delta = spark.read.format("delta").options(**adls_options).load(f"{GOLD_BASE}/{tabela}")
print(f"\n📂 Delta: {df_delta.count()} registros lidos")

save_gold_sqlserver_merge(
    df=df_delta,
    table_name=tabela,
    pk_cols="id_fornecedor",
)

In [0]:
# ── 7.5 — gold_lote_alerta_ativo_vencido (L6) — TRUNCATE+INSERT ────────
tabela = "gold_lote_alerta_ativo_vencido"
df_delta = spark.read.format("delta").options(**adls_options).load(f"{GOLD_BASE}/{tabela}")
print(f"\n📂 Delta: {df_delta.count()} registros lidos")

save_gold_sqlserver_overwrite(df=df_delta, table_name=tabela)

In [0]:
# ── 7.6 — gold_lote_volume_categoria_mes (L7) ──────────────────────────
tabela = "gold_lote_volume_categoria_mes"
df_delta = spark.read.format("delta").options(**adls_options).load(f"{GOLD_BASE}/{tabela}")
print(f"\n📂 Delta: {df_delta.count()} registros lidos")

save_gold_sqlserver_merge(
    df=df_delta,
    table_name=tabela,
    pk_cols=["canal", "categoria", "ano", "mes"],
)

In [0]:
# ── 7.7 — gold_lote_recall_fornecedor_trimestre (L8) ────────────────────
tabela = "gold_lote_recall_fornecedor_trimestre"
df_delta = spark.read.format("delta").options(**adls_options).load(f"{GOLD_BASE}/{tabela}")
print(f"\n📂 Delta: {df_delta.count()} registros lidos")

save_gold_sqlserver_merge(
    df=df_delta,
    table_name=tabela,
    pk_cols=["id_fornecedor", "ano", "trimestre"],
)

In [0]:
# ── 7.8 — gold_lote_shelf_life_categoria (L9) ──────────────────────────
tabela = "gold_lote_shelf_life_categoria"
df_delta = spark.read.format("delta").options(**adls_options).load(f"{GOLD_BASE}/{tabela}")
print(f"\n📂 Delta: {df_delta.count()} registros lidos")

save_gold_sqlserver_merge(
    df=df_delta,
    table_name=tabela,
    pk_cols="categoria_pai",
)

In [0]:
# ── 7.9 — gold_lote_sem_estoque (L10 fato) — TRUNCATE+INSERT ───────────
tabela = "gold_lote_sem_estoque"
df_delta = spark.read.format("delta").options(**adls_options).load(f"{GOLD_BASE}/{tabela}")
print(f"\n📂 Delta: {df_delta.count()} registros lidos")

save_gold_sqlserver_overwrite(df=df_delta, table_name=tabela)

In [0]:
# ── 7.10 — gold_fornecedor_canal_distribuicao (L10 dim) ────────────────
tabela = "gold_fornecedor_canal_distribuicao"
df_delta = spark.read.format("delta").options(**adls_options).load(f"{GOLD_BASE}/{tabela}")
print(f"\n📂 Delta: {df_delta.count()} registros lidos")

save_gold_sqlserver_merge(
    df=df_delta,
    table_name=tabela,
    pk_cols="id_fornecedor",
)

## BLOCO 8 — Validação final

Checa integridade da camada Gold após a gravação dupla (Delta + SQL Server):
1. **Paridade de contagem** entre Delta e SQL Server para cada tabela
2. **Unicidade de PK** no SQL Server (duplicata = bug no MERGE)
3. **Resumo de alertas** disparados (categorias sem fornecedor, lotes vencidos ativos etc.)

In [0]:
# ══════════════════════════════════════════════════════════════════════════
# BLOCO 8 — Validação final: Delta vs SQL Server + PK + alertas
# ══════════════════════════════════════════════════════════════════════════

print("\n" + "=" * 70)
print("🔍 BLOCO 8 — Validação final da Gold")
print("=" * 70)

jdbc_url_val = (
    f"jdbc:sqlserver://{SQL_HOST}:1433;"
    f"database={SQL_DB};encrypt=true;trustServerCertificate=false;"
    "hostNameInCertificate=*.database.windows.net;loginTimeout=30;"
)

def read_sql(query_sql):
    """Lê uma query do SQL Server via JDBC Reader."""
    return (
        spark.read.format("jdbc")
        .option("url", jdbc_url_val)
        .option("user", SQL_USER)
        .option("password", SQL_PASS)
        .option("driver", "com.microsoft.sqlserver.jdbc.SQLServerDriver")
        .option("query", query_sql)
        .load()
    )

# Catálogo das tabelas Gold + PKs
tabelas_gold = [
    ("gold_fornecedor_lead_time_trimestre",     ["tipo_fornecedor", "ano", "trimestre"]),
    ("gold_fornecedor_ativos_categoria",        ["categoria"]),
    ("gold_fornecedor_distribuicao_uf",         ["uf_origem"]),
    ("gold_fornecedor_tempo_ativacao",          ["id_fornecedor"]),
    ("gold_lote_alerta_ativo_vencido",          ["id_lote"]),
    ("gold_lote_volume_categoria_mes",          ["canal", "categoria", "ano", "mes"]),
    ("gold_lote_recall_fornecedor_trimestre",   ["id_fornecedor", "ano", "trimestre"]),
    ("gold_lote_shelf_life_categoria",          ["categoria_pai"]),
    ("gold_lote_sem_estoque",                   ["id_lote"]),
    ("gold_fornecedor_canal_distribuicao",      ["id_fornecedor"]),
]

# ── Validação 1 + 2: paridade de contagem e unicidade de PK ─────────────
print("\n── Paridade Delta ↔ SQL Server + Unicidade de PK ──\n")

resultado = []
for tabela, pk_cols in tabelas_gold:
    # Contagem Delta
    qtd_delta = (
        spark.read.format("delta").options(**adls_options)
        .load(f"{GOLD_BASE}/{tabela}")
        .count()
    )

    # Contagem SQL Server
    qtd_sql = read_sql(f"SELECT COUNT(*) AS n FROM squad3.{tabela}") \
        .collect()[0]["n"]

    # Checagem de PK duplicada no SQL
    pk_str = ", ".join(pk_cols)
    dup_query = f"""
        SELECT COUNT(*) AS n_duplicatas FROM (
            SELECT {pk_str}, COUNT(*) AS c
            FROM squad3.{tabela}
            GROUP BY {pk_str}
            HAVING COUNT(*) > 1
        ) t
    """
    qtd_duplicatas = read_sql(dup_query).collect()[0]["n_duplicatas"]

    paridade_ok = (qtd_delta == qtd_sql)
    pk_ok = (qtd_duplicatas == 0)
    status = "✅" if (paridade_ok and pk_ok) else "❌"

    resultado.append({
        "tabela": tabela,
        "delta": qtd_delta,
        "sql": qtd_sql,
        "duplicatas_pk": qtd_duplicatas,
        "status": status,
    })

    print(f"{status} {tabela:<45s} Delta={qtd_delta:>5}  SQL={qtd_sql:>5}  Dup={qtd_duplicatas}")

# ── Resumo da validação ────────────────────────────────────────────────
total_ok = sum(1 for r in resultado if r["status"] == "✅")
total_falha = len(resultado) - total_ok

print("\n" + "─" * 70)
print(f"📊 Resumo: {total_ok}/10 tabelas OK  |  {total_falha} com falha")
if total_falha > 0:
    print("\n❌ Tabelas com problemas:")
    for r in resultado:
        if r["status"] == "❌":
            motivo = []
            if r["delta"] != r["sql"]:
                motivo.append(f"paridade (Δ={r['delta'] - r['sql']})")
            if r["duplicatas_pk"] > 0:
                motivo.append(f"{r['duplicatas_pk']} chaves duplicadas")
            print(f"   {r['tabela']}: {' + '.join(motivo)}")

# ── Validação 3: resumo de alertas de negócio ──────────────────────────
print("\n" + "=" * 70)
print("🚨 Alertas de negócio disparados")
print("=" * 70)

# F8 — categorias sem fornecedor ativo
qtd_sem_ativo = read_sql("""
    SELECT COUNT(*) AS n FROM squad3.gold_fornecedor_ativos_categoria
    WHERE flag_alerta_sem_ativo = 1
""").collect()[0]["n"]

# F8 — categorias com risco de concentração (≤ 2 ativos)
qtd_risco_conc = read_sql("""
    SELECT COUNT(*) AS n FROM squad3.gold_fornecedor_ativos_categoria
    WHERE flag_risco_concentracao = 1
""").collect()[0]["n"]

# L6 — lotes ativos vencidos por severidade
sev_counts = read_sql("""
    SELECT severidade, COUNT(*) AS n
    FROM squad3.gold_lote_alerta_ativo_vencido
    GROUP BY severidade
""").collect()
sev_dict = {r["severidade"]: r["n"] for r in sev_counts}

# L10 — fornecedores atacado_b2b
qtd_b2b = read_sql("""
    SELECT COUNT(*) AS n FROM squad3.gold_fornecedor_canal_distribuicao
    WHERE perfil = 'atacado_b2b'
""").collect()[0]["n"]

# L10 — total de lotes sem estoque
qtd_sem_estoque = read_sql("""
    SELECT COUNT(*) AS n FROM squad3.gold_lote_sem_estoque
""").collect()[0]["n"]

print(f"\n📂 Fornecedores")
print(f"   • Categorias SEM nenhum fornecedor ativo:     {qtd_sem_ativo}")
print(f"   • Categorias com risco de concentração (≤2):  {qtd_risco_conc}")
print(f"   • Fornecedores perfil atacado_b2b:            {qtd_b2b}/60")

print(f"\n📦 Lotes")
print(f"   • Alertas 'Ativo vencido' — crítico (>180d):  {sev_dict.get('critico', 0)}")
print(f"   • Alertas 'Ativo vencido' — alto (31-180d):   {sev_dict.get('alto', 0)}")
print(f"   • Alertas 'Ativo vencido' — moderado (≤30d):  {sev_dict.get('moderado', 0)}")
print(f"   • Lotes sem registro em estoque:              {qtd_sem_estoque}")

# ── Mensagem final ─────────────────────────────────────────────────────
print("\n" + "=" * 70)
if total_falha == 0:
    print("🎉 INGESTÃO GOLD CONCLUÍDA COM SUCESSO")
    print("   10/10 tabelas sincronizadas entre Delta (ADLS) e SQL Server (squad3)")
    print("   Looker pronto para conectar em: internshipDatabase.squad3.gold_*")
else:
    print(f"⚠️  INGESTÃO COM {total_falha} FALHA(S) — investigar antes de usar Gold")
print("=" * 70)